In [ ]:
import os
import subprocess
import textwrap
from pathlib import Path

from anthropic import Anthropic, omit
from anthropic.types import Message, MessageParam, TextBlock
from dotenv import dotenv_values

env_path = Path.cwd().parent / ".env.template"
for key, ref in dotenv_values(env_path).items():
    if ref is None:
        continue
    os.environ[key] = subprocess.run(
        ["op", "read", ref], capture_output=True, text=True, check=True
    ).stdout.strip()

client = Anthropic(
    base_url="https://openrouter.ai/api",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
model = "deepseek/deepseek-v4.1-flash"

In [37]:
def get_reply(message: Message) -> str:
    text = next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "",
    )
    if not text:
        return f"[no text block; stop_reason={message.stop_reason}, content={message.content!r}]"
    return text


def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def wrap_text(text: str, width: int = 120) -> str:
    return "\n".join(textwrap.fill(line, width=width) for line in text.splitlines())


# Start with an empty message list
messages: list[MessageParam] = []

In [ ]:
def chat(messages: list[MessageParam], system: str | None = None) -> str:
    message = client.messages.create(
        model=model,
        max_tokens=10000,
        thinking={"type": "disabled"},
        messages=messages,
        system=system or omit,
    )
    return get_reply(message)

In [ ]:
system = "You are a scientific assistant which popularizes complex scientific concepts."

while True:
    # You: Is it true that quantum computers consume hugh amount of energy? Why is it so?
    # You: how is physical qubit made? of what, silicone?
    user_input = input("You: ")
    if user_input == "/exit":
        break

    add_user_message(messages, user_input)
    print(wrap_text(f"You: {user_input}"))

    reply = chat(messages, system=system)
    add_assistant_message(messages, reply)

    print(wrap_text(f"AI: {reply}"))
    print("\n---\n")

You: Is it true that quantum computers consume hugh amount of energy? Why is it so?
AI: The short answer: **it depends on what you compare them to, and how you measure "consumption."** The popular claim
that quantum computers "consume huge amounts of energy" is partly true but often oversimplified. Let me break it down.

## What actually consumes energy in a quantum computer

**1. The dilution refrigerator (the big one)**
Most superconducting quantum computers operate at ~10–20 millikelvin — colder than deep space. Keeping that cold
requires cryogenic systems that run continuously. A typical dilution fridge draws on the order of **10–20 kilowatts of
wall power**, essentially 24/7. That's roughly the draw of a few dozen homes.

**2. Control electronics**
Generating and routing microwave pulses to manipulate qubits, plus all the classical control hardware, adds substantial
power — often comparable to or larger than the fridge itself.

**3. Classical error correction (for fault-tolerant m